# Phase 1: Data Cleaning

Cleans `data/raw/titles.csv` and `data/raw/credits.csv`, documents every decision, and writes
the results to `data/cleaned/` for the SQL analysis phase. Nothing here silently drops rows —
every null-handling and filtering decision is explained inline before it's applied.

In [1]:
import ast
from pathlib import Path

import pandas as pd

RAW_DIR = Path("..") / "data" / "raw"
CLEANED_DIR = Path("..") / "data" / "cleaned"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

titles = pd.read_csv(RAW_DIR / "titles.csv")
credits = pd.read_csv(RAW_DIR / "credits.csv")

titles.shape, credits.shape

((5850, 15), (77801, 5))

## Decision 1 — Dedupe check on `id`

`titles.id` should be a unique primary key. Verified: **0 duplicate ids** out of 5,850 rows.
No action needed, but this integrity check is worth keeping in the pipeline — a future data
refresh could introduce dupes silently.

In [2]:
n_dupes = titles["id"].duplicated().sum()
assert n_dupes == 0, f"Expected no duplicate ids, found {n_dupes}"
print(f"Duplicate ids: {n_dupes} / {len(titles)}")

Duplicate ids: 0 / 5850


## Decision 2 — Explode `genres` and `production_countries`

Both columns are stringified Python list literals, e.g. `"['drama', 'crime']"`. Per-genre and
per-country aggregation (Phase 2 queries like "top genres by rating") need one row per
(title, genre) and (title, country) pair, not one row per title.

**Approach:** parse with `ast.literal_eval`, then explode into two long-format tables —
`titles_genres` (id, genre) and `titles_countries` (id, country) — keyed back to `titles.id`.
The original `genres`/`production_countries` columns stay on the main table for reference.

**Empty lists:** 59 titles have `genres == []` and 229 have `production_countries == []` — no
genre/country was recorded at all. These rows are dropped from the *exploded* tables only
(there's nothing to explode), but they remain in the main `titles` table so count-based
questions ('how many titles total') aren't affected.

In [3]:
def parse_list(value):
    parsed = ast.literal_eval(value)
    return parsed if isinstance(parsed, list) else []

titles["genres_list"] = titles["genres"].apply(parse_list)
titles["production_countries_list"] = titles["production_countries"].apply(parse_list)

titles_genres = (
    titles[["id", "genres_list"]]
    .explode("genres_list")
    .dropna(subset=["genres_list"])
    .rename(columns={"genres_list": "genre"})
    .reset_index(drop=True)
)

titles_countries = (
    titles[["id", "production_countries_list"]]
    .explode("production_countries_list")
    .dropna(subset=["production_countries_list"])
    .rename(columns={"production_countries_list": "country"})
    .reset_index(drop=True)
)

print(f"titles_genres: {len(titles_genres):,} rows covering {titles_genres['id'].nunique():,} titles")
print(f"titles_countries: {len(titles_countries):,} rows covering {titles_countries['id'].nunique():,} titles")

titles_genres: 15,088 rows covering 5,791 titles
titles_countries: 6,528 rows covering 5,621 titles


## Decision 3 — Nulls: `age_certification`, `seasons`, `imdb_score`/`tmdb_score`, `runtime`

**`seasons` (3,744 nulls, 64%):** null exactly whenever `type == 'MOVIE'` (verified below —
3,744 movies, 0 shows). This isn't missing data, it's a structural non-applicability — movies
don't have seasons. **Decision: leave as `NaN`**, don't fill with 0 (0 seasons would be a false
claim; NaN correctly means 'not applicable'). Any per-show seasons analysis should filter
`type == 'SHOW'` first, which naturally excludes these.

**`age_certification` (2,619 nulls, 45%):** no certification was ever recorded for these
titles — plausibly non-US/international titles or older catalog entries predating the
certification field. Dropping ~45% of rows is not viable. **Decision: fill with the explicit
label `'Not Rated'`** rather than leaving `NaN`, so groupby/count aggregations treat 'unknown
certification' as its own visible category instead of silently vanishing from group-bys (pandas
excludes NaN keys from `groupby` by default, which would understate totals without explanation).

**`imdb_score` (482 nulls) / `tmdb_score` (311 nulls):** two independent rating sources; a title
missing one often still has the other. **Decision: don't drop rows from the main table** — only
88 titles are missing *both* scores. Rating-based questions (Phase 2/3) should filter on
whichever score they're using (e.g. `WHERE imdb_score IS NOT NULL`); count-based questions
('how many titles added per year') keep every row. A `has_any_rating` flag is added for
convenience.

**`runtime == 0` (14 rows):** a runtime of exactly 0 minutes is not a real duration — it's almost
certainly an unrecorded value encoded as 0 rather than `NaN`. **Decision: keep the raw column
as-is** (no data is fabricated), but flag these via `runtime_is_missing` so the Phase 3
runtime-vs-rating correlation excludes them instead of treating 0 as a real short runtime.

In [4]:
# Verify seasons null <=> movie, before deciding to leave it as NaN
seasons_null_by_type = titles.groupby("type")["seasons"].apply(lambda s: s.isnull().sum())
print(seasons_null_by_type)
assert seasons_null_by_type.get("MOVIE", 0) == (titles["type"] == "MOVIE").sum()
assert seasons_null_by_type.get("SHOW", 0) == 0

titles["age_certification"] = titles["age_certification"].fillna("Not Rated")

titles["has_any_rating"] = ~(titles["imdb_score"].isnull() & titles["tmdb_score"].isnull())
print(f"\nTitles missing both imdb_score and tmdb_score: {(~titles['has_any_rating']).sum()}")

titles["runtime_is_missing"] = titles["runtime"] == 0
print(f"Titles with runtime == 0 (flagged as missing): {titles['runtime_is_missing'].sum()}")

type
MOVIE    3744
SHOW        0
Name: seasons, dtype: int64

Titles missing both imdb_score and tmdb_score: 88
Titles with runtime == 0 (flagged as missing): 14


## Decision 4 — `decade` column

`release_year` spans 1945–2022. A `decade` column (e.g. 1990, 2000, 2010) makes trend charts and
groupby's cleaner than binning by individual year in every query.

In [5]:
titles["decade"] = (titles["release_year"] // 10) * 10
titles[["release_year", "decade"]].drop_duplicates().sort_values("release_year").head()

,release_year,decade
0,1945,1940
13,1954,1950
23,1956,1950
14,1958,1950
31,1959,1950


## Decision 5 — `credits.csv` cleaning

**No full-row duplicates** (0 found) and **no id/person_id/role pairs that are true errors** —
173 rows fall into groups sharing the same `(id, person_id, role)` (88 of them beyond each
group's first occurrence), representing an actor credited under two different character names
on the same title (e.g. dubbed/uncredited variants, or playing two named roles). Inspected a
sample manually; these are legitimate distinct credits, not data entry errors, so **no rows are
dropped or merged**.

**`character` nulls (9,772 rows, ~13%):** all 4,550 `DIRECTOR` rows are null (expected —
directors don't have a character), plus 5,222 `ACTOR` rows (uncredited/unlisted character name
in the source data). **Decision: leave as `NaN`** — it's genuinely unknown, and dropping these
rows would incorrectly remove real actor/title credits from cast-based analysis (e.g. 'most
prolific actors') where the character name itself isn't needed.

In [6]:
print("Full duplicate rows:", credits.duplicated().sum())
print("Rows sharing (id, person_id, role):", credits.duplicated(subset=["id", "person_id", "role"], keep=False).sum())
print("\ncharacter nulls by role:")
print(credits.groupby("role")["character"].apply(lambda s: s.isnull().sum()))

credits_cleaned = credits.copy()

Full duplicate rows: 0
Rows sharing (id, person_id, role): 173

character nulls by role:
role
ACTOR       5222
DIRECTOR    4550
Name: character, dtype: int64


## Save cleaned outputs

Drop the intermediate list columns (kept only for exploding) and write four cleaned CSVs:
`titles_cleaned`, `titles_genres`, `titles_countries`, `credits_cleaned`.

In [7]:
titles_cleaned = titles.drop(columns=["genres_list", "production_countries_list"])

titles_cleaned.to_csv(CLEANED_DIR / "titles_cleaned.csv", index=False)
titles_genres.to_csv(CLEANED_DIR / "titles_genres.csv", index=False)
titles_countries.to_csv(CLEANED_DIR / "titles_countries.csv", index=False)
credits_cleaned.to_csv(CLEANED_DIR / "credits_cleaned.csv", index=False)

for name, df in {
    "titles_cleaned": titles_cleaned,
    "titles_genres": titles_genres,
    "titles_countries": titles_countries,
    "credits_cleaned": credits_cleaned,
}.items():
    print(f"{name}: {df.shape}")

titles_cleaned: (5850, 18)


titles_genres: (15088, 2)
titles_countries: (6528, 2)
credits_cleaned: (77801, 5)


## Load cleaned tables into SQLite

Adds the cleaned tables to `netflix.db` alongside the untouched raw `titles`/`credits` tables
from Phase 0, so Phase 2's SQL queries run against clean data while the raw tables remain
available for auditing.

In [8]:
import sqlite3

DB_PATH = Path("..") / "netflix.db"
conn = sqlite3.connect(DB_PATH)

titles_cleaned.to_sql("titles_cleaned", conn, if_exists="replace", index=False)
titles_genres.to_sql("titles_genres", conn, if_exists="replace", index=False)
titles_countries.to_sql("titles_countries", conn, if_exists="replace", index=False)
credits_cleaned.to_sql("credits_cleaned", conn, if_exists="replace", index=False)

tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
conn.close()
print("Tables in netflix.db:", [t[0] for t in tables])

Tables in netflix.db: ['titles', 'credits', 'titles_cleaned', 'titles_genres', 'titles_countries', 'credits_cleaned']
